<a href="https://colab.research.google.com/github/MadSlingshoter/Advance-machine-learning/blob/develop/AdvAILab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Advanced Machine Learning Lab 1

Mattias Bengtsson

Start by importing libraries.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
import torchvision.transforms as transforms
import torch.optim as optim

In [ ]:
root_dir = "/content/data"

## Hyperparameters

Smaller learning rates are slower, but no significant difference in accuracy.

Larger batch sizes give only slightly higher accuracy.

The number of epochs don't see an increase in accuracy past about 7.

In [ ]:
# the stepsize of the updates of the model
learning_rate = 1e-3
# number of samples in each epoch
batch_size = 100
# number of iterations over the dataset
epochs = 7

The dataset used is the CIFAR-10 dataset of 60000 32x32 color images in 10 classes. There are 50000 training images and 10000 test images. It is included in torchvision, but it can also be found at https://www.cs.toronto.edu/~kriz/cifar.html

This is a classification problem as it is about correctly categorising images.

The output of torchvision datasets are PILImage images of range [0, 1]. We transform them to Tensors of normalized range [-1, 1].

Dividing the data into training and test datasets.

Dataloader handles the data manipulation; such as batches, shuffling data for each epoch, using multiprocessing, etc.; for us.

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

training_data = datasets.CIFAR10(
    root=root_dir,
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.CIFAR10(
    root=root_dir,
    train=False,
    download=True,
    transform=transform
)

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# May need to change runtime type hardware accelerator to T4 GPU if running in colab (personal note)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


Data is already flattened. The first 1024 entries contain the red channel values, the next 1024 the green, and the final 1024 the blue. The image is stored in row-major order, so that the first 32 entries of the array are the red channel values of the first row of the image.

Therefore, the input width is 3072 and the input height is 1.

5 linear layers with ReLU activation function between each.

In [ ]:
# Defining the Neural Network with 5 layers.
class NeuralNetwork(nn.Module):
    def __init__(self, input_width, input_height, label_dim):
        super(NeuralNetwork, self).__init__()
        self.linear_stack = nn.Sequential(
            nn.Linear(input_width*input_height, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, label_dim)
        )
        self.softmax = nn.Softmax(dim=1)
        self.softmax_result = 0

    def forward(self, x):
        x = torch.flatten(x, start_dim=1) #self.flatten(x) did not work
        logits = self.linear_stack(x)
        self.softmax_result = self.softmax(logits)
        return logits

In [ ]:
model = NeuralNetwork(3072, 1, 10).to(device)
print(model)

NeuralNetwork(
  (linear_stack): Sequential(
    (0): Linear(in_features=3072, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): ReLU()
    (8): Linear(in_features=512, out_features=10, bias=True)
  )
  (softmax): Softmax(dim=1)
)


Defining the train and test loops.

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        # X.shape = [64, 3072, 1] [batch_size, width, height]
        # y.shape = [64] [batch_size]
        X, y = X.to(device), y.to(device) # everything in the same device

        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad() #Reset the gradient
        loss.backward() # Calculate the gradient
        optimizer.step() # Correct the weights

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad(): # Context-manager that disables gradient calculation
        for X, y in dataloader:
            X, y = X.to(device), y.to(device) # everything in the same device
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    # sum up the losses and average by the number of batches
    # sum up all the correct predictions and find the accuracy ((TP+TN)/all)
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The loss function calculates the deviation of the model's predictions from the ground truth. Here we use cross entropy loss

$L = - \sum_{k=1}^Ky_k\log(p_k)$

The optimizer is used to minimize the loss function. Here we use AdamW. Stochastic Gradient Descent only gave about 20% accuracy. Even though the slightly above 50% we got is not high, it is better than the 10% we would have gotten from just random guessing.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.305069  [    0/50000]
loss: 1.532449  [10000/50000]
loss: 1.776559  [20000/50000]
loss: 1.745607  [30000/50000]
loss: 1.689383  [40000/50000]
Test Error: 
 Accuracy: 46.7%, Avg loss: 1.531563 

Epoch 2
-------------------------------
loss: 1.695741  [    0/50000]
loss: 1.730809  [10000/50000]
loss: 1.551816  [20000/50000]
loss: 1.742173  [30000/50000]
loss: 1.313042  [40000/50000]
Test Error: 
 Accuracy: 49.4%, Avg loss: 1.444433 

Epoch 3
-------------------------------
loss: 1.382425  [    0/50000]
loss: 1.449621  [10000/50000]
loss: 1.551581  [20000/50000]
loss: 1.480751  [30000/50000]
loss: 1.284037  [40000/50000]
Test Error: 
 Accuracy: 51.5%, Avg loss: 1.390857 

Epoch 4
-------------------------------
loss: 1.243322  [    0/50000]
loss: 1.225929  [10000/50000]
loss: 1.299913  [20000/50000]
loss: 1.377322  [30000/50000]
loss: 1.380149  [40000/50000]
Test Error: 
 Accuracy: 52.3%, Avg loss: 1.373454 

Epoch 5
------------------------

For interest, checking how accurate the model is for each class.

In [ ]:
# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in test_dataloader:
        images, labels = data[0].to(device), data[1].to(device) # everything on same device
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

Accuracy for class: plane is 56.8 %
Accuracy for class: car   is 70.5 %
Accuracy for class: bird  is 37.0 %
Accuracy for class: cat   is 35.2 %
Accuracy for class: deer  is 47.0 %
Accuracy for class: dog   is 41.9 %
Accuracy for class: frog  is 69.3 %
Accuracy for class: horse is 57.3 %
Accuracy for class: ship  is 69.7 %
Accuracy for class: truck is 54.2 %
